# Chapter 5 — Exploratory Data Analysis and Data Quality
## 5.13 End-to-End Insurance Cancellation Case Study

**Source note:** This notebook follows the available Chapter 5 lesson content, reorganized for notebook study; it is not a word-for-word export of the conversation.

**Notebook use:** Read the explanations, run the code cells in order, interpret the results, and complete the exercises.

### 5.13.1 Business framing

Predict policy cancellation within 90 days. Define the observation as a policy at a monthly snapshot, and use only features available on that snapshot date. Risk ranking is not the same as uplift or causal effect of outreach.

### 5.13.2 Synthetic case study

In [ ]:
import numpy as np,pandas as pd
rng=np.random.default_rng(42);n=2500
dates=pd.date_range("2024-01-01",periods=24,freq="MS")
rows=[]
for d in dates:
    tenure=rng.uniform(.5,12,n);late=rng.poisson(.5,n);calls=rng.poisson(1.5,n)
    premium=rng.lognormal(7.4,.45,n)
    p=1/(1+np.exp(-(-3+.35*late+.25*calls-.1*tenure+.00012*premium)))
    rows.append(pd.DataFrame({"snapshot_date":d,"tenure":tenure,"late_payments":late,"service_calls":calls,"premium":premium,"cancelled_90d":rng.binomial(1,p)}))
df=pd.concat(rows,ignore_index=True)

### 5.13.3 Temporal evaluation and label maturity

In [ ]:
train=df[df.snapshot_date<="2024-12-01"].copy()
val=df[(df.snapshot_date>="2025-04-01")&(df.snapshot_date<="2025-06-01")].copy()
test=df[df.snapshot_date>="2025-10-01"].copy()
print(len(train),len(val),len(test))

### 5.13.4 Fit and compare models

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score,roc_auc_score
cols=["tenure","late_payments","service_calls","premium"]
models={"logistic":make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),"forest":RandomForestClassifier(n_estimators=100,min_samples_leaf=10,random_state=42)}
for name,model in models.items():
    model.fit(train[cols],train.cancelled_90d)
    s=model.predict_proba(val[cols])[:,1]
    print(name,"ROC-AUC",roc_auc_score(val.cancelled_90d,s),"AP",average_precision_score(val.cancelled_90d,s))

### 5.13.5 Thresholds and value

For outreach cost $C$, retention value $V$, and intervention success probability $u$, a simplified expected value is $EV_i=\hat p_i uV-C$. **Predictive risk is not an estimate of causal uplift.**

### 5.13.6 Limitations

This instructional snapshot simulation does not generate internally consistent cancellation event histories or account for censoring. Real labels must come from actual cancellation dates, and correlated policy-month observations require care.

### 5.13.7 Practice

Choose a model using validation AP, evaluate once on test, and propose a capacity-constrained top-K outreach policy.

### Review and reflection

- What did this section establish?
- Which assumptions matter?
- How would you use the technique in an unfamiliar real-world dataset?
- What could go wrong if it is applied incorrectly?